In [1]:
import pandas as pd

In [2]:
df = pd.read_csv("IMDB Dataset.csv")

In [3]:
df.head()

,review,sentiment
0,One of the other reviewers has mentioned that ...,positive
1,A wonderful little production. <br /><br />The...,positive
2,I thought this was a wonderful way to spend ti...,positive
3,Basically there's a family where a little boy ...,negative
4,"Petter Mattei's ""Love in the Time of Money"" is...",positive


In [4]:
df.tail()

,review,sentiment
49995,I thought this movie did a down right good job...,positive
49996,"Bad plot, bad dialogue, bad acting, idiotic di...",negative
49997,I am a Catholic taught in parochial elementary...,negative
49998,I'm going to have to disagree with the previou...,negative
49999,No one expects the Star Trek movies to be high...,negative


In [5]:
df.isnull().sum()

review       0
sentiment    0
dtype: int64

In [6]:
df.shape

(50000, 2)

In [7]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 50000 entries, 0 to 49999
Data columns (total 2 columns):
 #   Column     Non-Null Count  Dtype
---  ------     --------------  -----
 0   review     50000 non-null  str  
 1   sentiment  50000 non-null  str  
dtypes: str(2)
memory usage: 781.4 KB


In [8]:
df.duplicated().sum()

np.int64(418)

In [11]:
df = df.drop_duplicates()

In [12]:
df.duplicated().sum()

np.int64(0)

In [14]:
print(df['sentiment'].value_counts())

sentiment
positive    24884
negative    24698
Name: count, dtype: int64


In [15]:
print(df['review'].iloc[0])

One of the other reviewers has mentioned that after watching just 1 Oz episode you'll be hooked. They are right, as this is exactly what happened with me.<br /><br />The first thing that struck me about Oz was its brutality and unflinching scenes of violence, which set in right from the word GO. Trust me, this is not a show for the faint hearted or timid. This show pulls no punches with regards to drugs, sex or violence. Its is hardcore, in the classic use of the word.<br /><br />It is called OZ as that is the nickname given to the Oswald Maximum Security State Penitentary. It focuses mainly on Emerald City, an experimental section of the prison where all the cells have glass fronts and face inwards, so privacy is not high on the agenda. Em City is home to many..Aryans, Muslims, gangstas, Latinos, Christians, Italians, Irish and more....so scuffles, death stares, dodgy dealings and shady agreements are never far away.<br /><br />I would say the main appeal of the show is due to the fac

In [16]:
from sklearn.model_selection import train_test_split
X = df['review']
y = df['sentiment']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.2, random_state = 42, stratify = y)

print("Training data: ", X_train.shape)
print("Testing data: ", X_test.shape)

Training data:  (39665,)
Testing data:  (9917,)


In [ ]:
import re
import nltk
from nltk import word_tokenize
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from sklearn.feature_extraction.text import TfidfVectorizer

nltk.download("punkt_tab")
nltk.download("stopwords")
nltk.download("wordnet")

stop_words = set(stopwords.words("english"))
lemmatizer = WordNetLemmatizer()

def clean_text(text):
    text = text.lower()
    text = re.sub(r'<br\s*/?>', '', text)
    text = re.sub(r'[^a-zA-Z\s]', '', text)
    
    words = word_tokenize(text)
    cleaned_words = [lemmatizer.lemmatize(i) for i in words if i not in stop_words]
    return " ".join(cleaned_words)

X_train_cleaned = X_train.apply(clean_text)
X_test_cleaned = X_test.apply(clean_text)

vectorizer = TfidfVectorizer()
X_train_vec = vectorizer.fit_transform(X_train_cleaned)
X_test_vec = vectorizer.transform(X_test_cleaned)

print("Train vectors shape:", X_train_vec.shape)
print("Test vectors shape:", X_test_vec.shape)

In [18]:
from sklearn.linear_model import LogisticRegression
model = LogisticRegression(max_iter = 1000)
model.fit(X_train_vec, y_train)
print("Model Training complete.")

Model Training complete.


In [21]:
from sklearn.metrics import accuracy_score, classification_report

y_pred = model.predict(X_test_vec)
accuracy = accuracy_score(y_test, y_pred)
print(f"Overall Model Accuracy: {accuracy:.2%}\n")

print("Detailed Classification Report:")
print(classification_report(y_test, y_pred))


Overall Model Accuracy: 88.61%

Detailed Classification Report:
              precision    recall  f1-score   support

    negative       0.90      0.87      0.88      4940
    positive       0.88      0.90      0.89      4977

    accuracy                           0.89      9917
   macro avg       0.89      0.89      0.89      9917
weighted avg       0.89      0.89      0.89      9917



In [22]:
def predict_my_review(custom_review):
    cleaned = clean_text(custom_review)
    vectorized = vectorizer.transform([cleaned])
    prediction = model.predict(vectorized)[0]
    probabilities = model.predict_proba(vectorized)[0]
    confidence = max(probabilities) * 100
    
    print(f"Review: \"{custom_review}\"")
    print(f"Predicted Sentiment: {prediction.upper()} ({confidence:.2f}% confidence)\n")

predict_my_review("This movie was an absolute masterpiece. The acting was incredible and the plot kept me on the edge of my seat!")
predict_my_review("What a complete waste of time. The script was terrible and the characters were so boring.")


Review: "This movie was an absolute masterpiece. The acting was incredible and the plot kept me on the edge of my seat!"
Predicted Sentiment: POSITIVE (89.16% confidence)

Review: "What a complete waste of time. The script was terrible and the characters were so boring."
Predicted Sentiment: NEGATIVE (100.00% confidence)

